In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%pip install -q -U unsloth trl datasets bitsandbytes accelerate

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.7/75.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 130.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import json
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_CONNECT_TIMEOUT"] = "60"
os.environ["HF_HUB_READ_TIMEOUT"] = "60"

REPO_ID = "uitnlp/OpenViVQA-dataset"
DATA_ROOT = Path("/content/openvivqa-data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def prepare_split(split):
    json_name = f"vlsp2023_{split}_data.json"
    zip_name = f"{split}-images.zip"

    json_path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=json_name,
        cache_dir="/content/hf-cache",
    )

    image_root = DATA_ROOT / split
    image_dir = image_root / f"{split}-images"
    image_root.mkdir(parents=True, exist_ok=True)

    if not image_dir.exists() or not any(image_dir.glob("*.jpg")):
        archive = hf_hub_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            filename=zip_name,
            cache_dir="/content/hf-cache",
        )

        with zipfile.ZipFile(archive) as zf:
            zf.extractall(image_root)

    print(split, "images:", len(list(image_dir.glob("*.jpg"))))
    return json_path, image_dir

train_json, train_images = prepare_split("train")
dev_json, dev_images = prepare_split("dev")

vlsp2023_train_data.json:   0%|          | 0.00/6.60M [00:00<?, ?B/s]

train-images.zip: reconstructing file:   0%|          |  0.00B / 1.34GB            

train-images.zip: downloading bytes:           |  0.00B            

train images: 0


vlsp2023_dev_data.json:   0%|          | 0.00/762k [00:00<?, ?B/s]

dev-images.zip: reconstructing file:   0%|          |  0.00B /  165MB            

dev-images.zip: downloading bytes:           |  0.00B            

dev images: 1070


In [ ]:
from datasets import Dataset

def load_records(json_path, image_dir):
    raw = json.loads(Path(json_path).read_text(encoding="utf-8"))

    images = raw["images"]
    annotations = raw["annotations"]
    records = []

    items = (
        annotations.items()
        if isinstance(annotations, dict)
        else enumerate(annotations)
    )

    for _, ann in items:
        image_id = ann["image_id"]
        filename = images.get(str(image_id)) or images.get(image_id)

        if not filename:
            continue

        image_path = image_dir / filename

        if image_path.exists():
            records.append({
                "question": ann["question"],
                "answer": ann["answer"],
                "image_path": str(image_path),
            })

    print("Samples:", len(records))
    return records

train_records = load_records(train_json, train_images)
dev_records = load_records(dev_json, dev_images)

Samples: 0
Samples: 3545


In [ ]:
import gc
import torch
from unsloth import FastLanguageModel
from transformers import AutoProcessor

# Giải phóng model nếu cell trước đã nạp dở
for name in ["model", "tokenizer"]:
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "unsloth/Qwen2-VL-2B-Instruct"

model, _ = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048,
    load_in_4bit=True,
)

# Đặt giới hạn ảnh ngay lúc khởi tạo processor
tokenizer = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=200704,
    max_pixels=401408,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=True,
    loftq_config=None,
)

print("Đã nạp Qwen2-VL-2B và gắn LoRA.")

==((====))==  Unsloth 2026.8.18: Fast Qwen2_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found


preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Đã nạp Qwen2-VL-2B và gắn LoRA.


In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download
import zipfile

TRAIN_ROOT = Path("/content/openvivqa-data/train")
TRAIN_ROOT.mkdir(parents=True, exist_ok=True)

# Tìm thư mục đã giải nén trước đó
candidates = [
    TRAIN_ROOT / "training-images",
    TRAIN_ROOT / "train-images",
]

train_images = next(
    (
        p for p in candidates
        if p.exists() and any(p.rglob("*.jpg"))
    ),
    None,
)

# Nếu chưa có thì tải đúng file training-images.zip
if train_images is None:
    archive = hf_hub_download(
        repo_id="uitnlp/OpenViVQA-dataset",
        repo_type="dataset",
        filename="training-images.zip",
        cache_dir="/content/hf-cache",
    )

    with zipfile.ZipFile(archive) as zf:
        zf.extractall(TRAIN_ROOT)

    train_images = TRAIN_ROOT / "training-images"

print("Thư mục train:", train_images)
print("Số ảnh train:", len(list(train_images.rglob("*.jpg"))))

Thư mục train: /content/openvivqa-data/train/training-images
Số ảnh train: 9129


In [ ]:
train_records = load_records(
    train_json,
    train_images,
)

dev_records = load_records(
    dev_json,
    dev_images,
)

print("Train:", len(train_records))
print("Dev:", len(dev_records))

assert len(train_records) > 0
assert len(dev_records) > 0

Train: 30833
Dev: 3545


In [ ]:
from pathlib import Path
from transformers import TrainingArguments
from trl import SFTTrainer
from trl.trainer.sft_trainer import (
    DataCollatorForVisionLanguageModeling,
)

# Dùng thư mục mới, không trộn checkpoint 3 epoch cũ
OUTPUT_DIR = (
    "/content/drive/MyDrive/TtI/"
    "Vi-VQA-End-to-End/outputs-2B-supervised-quick"
)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model.config.use_cache = False

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,

        # Bản rút gọn để giảm thời gian
        max_steps=1800,
        num_train_epochs=1,

        learning_rate=1e-5,
        warmup_steps=10,
        weight_decay=0.01,

        fp16=True,
        bf16=False,
        optim="adamw_8bit",

        logging_steps=10,
        eval_strategy="no",

        save_strategy="steps",
        save_steps=300,
        save_total_limit=2,

        output_dir=OUTPUT_DIR,
        remove_unused_columns=False,
        report_to=[],
    ),
)

trainer.data_collator = DataCollatorForVisionLanguageModeling(
    processor=tokenizer,
    max_length=1024,
)

print("Đã tạo trainer 2B fine-tune rút gọn.")
print("Output:", OUTPUT_DIR)

Đã tạo trainer 2B fine-tune rút gọn.
Output: /content/drive/MyDrive/TtI/Vi-VQA-End-to-End/outputs-2B-supervised-quick


In [ ]:
result = trainer.train()

print("Train loss:", result.training_loss)

final_dir = f"{OUTPUT_DIR}/final_lora"

trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

print("Đã lưu adapter tại:")
print(final_dir)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 30,833 | Num Epochs = 1 | Total steps = 1,800
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 73,859,072 of 2,282,844,672 (3.24% trained)


Step,Training Loss
10,7.220526
20,7.206946
30,7.195799
40,7.193110
50,7.184903
60,7.210572
70,7.195744
80,7.189806
90,7.200562
100,7.185226
